# Robustesse de la détection d'objets face à la météo — YOLOv8 et RT-DETR

Images de caméras de contrôle routier (5 classes : plaque, téléphone, véhicule, ceinture, pare-brise),
dégradées par 5 conditions météo (brouillard, nuit, pluie, neige, éblouissement) à 5 niveaux.

**Protocole :** on entraîne le modèle sur les images **originales** uniquement, puis on le teste sur les
**mêmes images de test** dans chacune des 26 conditions. La baisse de mAP mesure la robustesse.

Avant de commencer : *Exécution > Modifier le type d'exécution > GPU T4*.

In [ ]:
!pip install -q ultralytics
import os, random, glob, zipfile
import pandas as pd
from ultralytics import YOLO

## 1. Récupérer le dataset depuis Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP = '/content/drive/MyDrive/results.zip'   # le zip du dossier results, posé à la racine de ton Drive
if not os.path.exists('/content/results'):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('/content')
DATA = '/content/results'
CONDITIONS = sorted(os.listdir(DATA))

# YOLO cherche les labels dans un dossier « labels » à côté de « images »
for c in CONDITIONS:
    if os.path.exists(f'{DATA}/{c}/labels_yolo') and not os.path.exists(f'{DATA}/{c}/labels'):
        os.rename(f'{DATA}/{c}/labels_yolo', f'{DATA}/{c}/labels')
print(len(CONDITIONS), 'conditions :', CONDITIONS)


## 2. Découpage train / val / test (70 / 15 / 15, graine 42)
Le découpage est fait une seule fois sur les images originales, puis réutilisé pour toutes les conditions.

In [ ]:
CLASSES = ['license_plate', 'phone', 'vehicle', 'seatbelt', 'windshield']
random.seed(42)
noms = sorted(os.listdir(f'{DATA}/original/images'))
random.shuffle(noms)
n = len(noms)
split = {'train': noms[:int(.7*n)], 'val': noms[int(.7*n):int(.85*n)], 'test': noms[int(.85*n):]}
print({k: len(v) for k, v in split.items()})

os.makedirs('/content/splits', exist_ok=True)
def ecrire_liste(condition, partie):
    chemin = f'/content/splits/{condition}_{partie}.txt'
    with open(chemin, 'w') as f:
        for nom in split[partie]:
            img = f'{DATA}/{condition}/images/{nom}'
            if os.path.exists(img):
                f.write(img + '\n')
    return chemin

def ecrire_yaml(nom, train, val):
    chemin = f'/content/splits/{nom}.yaml'
    with open(chemin, 'w') as f:
        f.write(f'train: {train}\nval: {val}\nnc: {len(CLASSES)}\nnames: {CLASSES}\n')
    return chemin

yaml_entrainement = ecrire_yaml('original', ecrire_liste('original', 'train'), ecrire_liste('original', 'val'))

## 3. Entraîner YOLOv8 sur les images originales

In [ ]:
modele = YOLO('yolov8s.pt')
modele.train(data=yaml_entrainement, epochs=50, imgsz=640, batch=16, seed=42,
             project='/content/drive/MyDrive/robustesse_meteo', name='yolov8s', exist_ok=True)

## 4. Tester le modèle sur les 26 conditions

In [ ]:
meilleur = YOLO('/content/drive/MyDrive/robustesse_meteo/yolov8s/weights/best.pt')
resultats = []
for c in CONDITIONS:
    y = ecrire_yaml(f'test_{c}', ecrire_liste('original', 'train'), ecrire_liste(c, 'test'))
    m = meilleur.val(data=y, imgsz=640, batch=16, verbose=False, plots=False)
    meteo, niveau = ('original', 0) if c == 'original' else (c.split('_level')[0], int(c.split('_level')[1]))
    resultats.append({'modele': 'YOLOv8s', 'meteo': meteo, 'niveau': niveau,
                      'mAP50': m.box.map50, 'mAP50_95': m.box.map})
    print(c, round(m.box.map50, 3), round(m.box.map, 3))

df = pd.DataFrame(resultats)
df.to_csv('/content/drive/MyDrive/robustesse_meteo/resultats_yolov8s.csv', index=False)
df

## 5. Visualiser la baisse de performance

In [ ]:
import matplotlib.pyplot as plt
ref = df[df.meteo == 'original'].mAP50.iloc[0]
plt.figure(figsize=(8, 5))
for meteo, g in df[df.meteo != 'original'].groupby('meteo'):
    g = g.sort_values('niveau')
    plt.plot([0] + list(g.niveau), [ref] + list(g.mAP50), marker='o', label=meteo)
plt.xlabel('Niveau de dégradation (0 = image originale)')
plt.ylabel('mAP@50')
plt.title('YOLOv8s : robustesse face à la météo')
plt.legend(); plt.grid(alpha=.3)
plt.savefig('/content/drive/MyDrive/robustesse_meteo/courbe_yolov8s.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Deuxième modèle : RT-DETR (transformer)
Même protocole : entraînement sur les originales, test sur les 26 conditions.

In [ ]:
from ultralytics import RTDETR

rtdetr = RTDETR('rtdetr-l.pt')
rtdetr.train(data=yaml_entrainement, epochs=50, imgsz=640, batch=8, seed=42,
             project='/content/drive/MyDrive/robustesse_meteo', name='rtdetr-l', exist_ok=True)

meilleur_rtdetr = RTDETR('/content/drive/MyDrive/robustesse_meteo/rtdetr-l/weights/best.pt')
resultats_rtdetr = []
for c in CONDITIONS:
    y = ecrire_yaml(f'test_{c}', ecrire_liste('original', 'train'), ecrire_liste(c, 'test'))
    m = meilleur_rtdetr.val(data=y, imgsz=640, batch=8, verbose=False, plots=False)
    meteo, niveau = ('original', 0) if c == 'original' else (c.split('_level')[0], int(c.split('_level')[1]))
    resultats_rtdetr.append({'modele': 'RT-DETR-l', 'meteo': meteo, 'niveau': niveau,
                             'mAP50': m.box.map50, 'mAP50_95': m.box.map})
    print(c, round(m.box.map50, 3), round(m.box.map, 3))

df_rtdetr = pd.DataFrame(resultats_rtdetr)
df_rtdetr.to_csv('/content/drive/MyDrive/robustesse_meteo/resultats_rtdetr-l.csv', index=False)
df_rtdetr

## 7. Comparer les deux modèles

In [ ]:
tous = pd.concat([pd.read_csv('/content/drive/MyDrive/robustesse_meteo/resultats_yolov8s.csv'), df_rtdetr])
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=True)
for ax, meteo in zip(axes, ['fog', 'night', 'rain', 'snow', 'glare']):
    for modele, g in tous.groupby('modele'):
        ref = g[g.meteo == 'original'].mAP50.iloc[0]
        g = g[g.meteo == meteo].sort_values('niveau')
        ax.plot([0] + list(g.niveau), [ref] + list(g.mAP50), marker='o', label=modele)
    ax.set_title(meteo); ax.set_xlabel('Niveau'); ax.grid(alpha=.3)
axes[0].set_ylabel('mAP@50'); axes[0].legend()
plt.suptitle('YOLOv8s contre RT-DETR-l face à la météo')
plt.savefig('/content/drive/MyDrive/robustesse_meteo/comparaison_modeles.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Troisième modèle : Faster R-CNN (deux étapes, torchvision)
Même découpage et mêmes images de test. Entraînement en PyTorch pur (20 époques), mAP calculé avec torchmetrics (protocole COCO).

In [ ]:
!pip install -q torchmetrics pycocotools
import time, copy, torch
from PIL import Image
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torchvision.transforms.functional as TF
from torchmetrics.detection import MeanAveragePrecision

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DOSSIER_FRCNN = '/content/drive/MyDrive/robustesse_meteo/faster-rcnn'
os.makedirs(DOSSIER_FRCNN, exist_ok=True)

class ImagesYOLO(torch.utils.data.Dataset):
    """Lit les images d'une condition et convertit les labels YOLO en boîtes (x1, y1, x2, y2) en pixels."""
    def __init__(self, condition, partie, retourner=False):
        self.images = [f'{DATA}/{condition}/images/{n}' for n in split[partie]
                       if os.path.exists(f'{DATA}/{condition}/images/{n}')]
        self.retourner = retourner
    def __len__(self):
        return len(self.images)
    def __getitem__(self, i):
        chemin = self.images[i]
        img = Image.open(chemin).convert('RGB')
        L, H = img.size
        label = os.path.splitext(chemin.replace('/images/', '/labels/'))[0] + '.txt'
        boites, classes = [], []
        if os.path.exists(label):
            for ligne in open(label):
                p = ligne.split()
                if len(p) < 5:
                    continue
                c, x, y, w, h = int(p[0]), *map(float, p[1:5])
                x1, y1, x2, y2 = (x - w/2) * L, (y - h/2) * H, (x + w/2) * L, (y + h/2) * H
                if x2 - x1 > 1 and y2 - y1 > 1:
                    boites.append([x1, y1, x2, y2]); classes.append(c + 1)   # 0 = fond pour Faster R-CNN
        img = TF.to_tensor(img)
        boites = torch.tensor(boites, dtype=torch.float32).reshape(-1, 4)
        if self.retourner and torch.rand(1) < 0.5:
            img = img.flip(-1)
            boites[:, [0, 2]] = L - boites[:, [2, 0]]
        return img, {'boxes': boites, 'labels': torch.tensor(classes, dtype=torch.int64)}

def regrouper(lot):
    return tuple(zip(*lot))

def chargeur(condition, partie, retourner=False, melanger=False):
    return torch.utils.data.DataLoader(ImagesYOLO(condition, partie, retourner), batch_size=4,
                                       shuffle=melanger, num_workers=2, collate_fn=regrouper)

@torch.no_grad()
def evaluer(modele, condition, partie='test'):
    modele.eval()
    mesure = MeanAveragePrecision(iou_type='bbox')
    for imgs, cibles in chargeur(condition, partie):
        sorties = modele([im.to(device) for im in imgs])
        mesure.update([{k: v.cpu() for k, v in s.items()} for s in sorties], list(cibles))
    r = mesure.compute()
    return r['map_50'].item(), r['map'].item()

# Modèle pré-entraîné sur COCO, dernière couche remplacée pour nos 5 classes (+ le fond)
frcnn = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT)
frcnn.roi_heads.box_predictor = FastRCNNPredictor(frcnn.roi_heads.box_predictor.cls_score.in_features, len(CLASSES) + 1)
frcnn.to(device)

EPOQUES = 20
params = [p for p in frcnn.parameters() if p.requires_grad]
optimiseur = torch.optim.SGD(params, lr=0.01, momentum=0.9, weight_decay=1e-4)
planning = torch.optim.lr_scheduler.MultiStepLR(optimiseur, milestones=[15], gamma=0.1)
echelle = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
entrainement = chargeur('original', 'train', retourner=True, melanger=True)

meilleur_score, meilleurs_poids = -1, None
debut = time.time()
for epoque in range(1, EPOQUES + 1):
    frcnn.train()
    total = 0
    for imgs, cibles in entrainement:
        imgs = [im.to(device) for im in imgs]
        cibles = [{k: v.to(device) for k, v in t.items()} for t in cibles]
        with torch.autocast(device_type=device.type, enabled=device.type == 'cuda'):
            pertes = sum(frcnn(imgs, cibles).values())
        optimiseur.zero_grad()
        echelle.scale(pertes).backward()
        echelle.step(optimiseur)
        echelle.update()
        total += pertes.item()
    planning.step()
    map50, map5095 = evaluer(frcnn, 'original', 'val')
    print(f'Époque {epoque}/{EPOQUES}  perte {total / len(entrainement):.3f}  val mAP50 {map50:.3f}  mAP50-95 {map5095:.3f}')
    if map50 > meilleur_score:
        meilleur_score, meilleurs_poids = map50, copy.deepcopy(frcnn.state_dict())
print(f'Entraînement terminé en {(time.time() - debut) / 60:.1f} min, meilleur mAP50 de validation : {meilleur_score:.3f}')

frcnn.load_state_dict(meilleurs_poids)
torch.save(meilleurs_poids, f'{DOSSIER_FRCNN}/best.pt')

# Vitesse d'inférence (ms par image)
frcnn.eval()
imgs, _ = next(iter(chargeur('original', 'test')))
with torch.no_grad():
    frcnn([imgs[0].to(device)])
    if device.type == 'cuda': torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(20):
        frcnn([imgs[0].to(device)])
    if device.type == 'cuda': torch.cuda.synchronize()
print(f'Inférence : {(time.time() - t0) / 20 * 1000:.0f} ms par image')

resultats_frcnn = []
for c in CONDITIONS:
    map50, map5095 = evaluer(frcnn, c, 'test')
    meteo, niveau = ('original', 0) if c == 'original' else (c.split('_level')[0], int(c.split('_level')[1]))
    resultats_frcnn.append({'modele': 'Faster R-CNN', 'meteo': meteo, 'niveau': niveau,
                            'mAP50': map50, 'mAP50_95': map5095})
    print(c, round(map50, 3), round(map5095, 3))

df_frcnn = pd.DataFrame(resultats_frcnn)
df_frcnn.to_csv('/content/drive/MyDrive/robustesse_meteo/resultats_faster-rcnn.csv', index=False)
df_frcnn
